# CALM-Sep Stage 4: Joint Fine-tuning

Fine-tune all components jointly: SR-CorrNet LoRA adapters + gate + Level-2 analyzer.

**LR = Stage 1 LR / 10** = **1e-5** (BLUEPRINT §9.2).
Optional: `--use-olora` enables O-LoRA penalty to prevent cross-adapter interference.

Prerequisites:
- Stage 1 adapters in `STAGE1_DIR`.
- Stage 3 gate + level2 in `STAGE3_DIR`.

Expected time: 15–20 epochs × ~20 min/epoch ≈ **5–7 hours** on T4.

In [ ]:
DATA_ROOT      = "/kaggle/input/calmsep-8k"
CHECKPOINT_DIR = "/kaggle/working/joint"
STAGE1_DIR     = "/kaggle/working/adapters"
STAGE3_DIR     = "/kaggle/working/gate"
EPOCHS         = 20
BATCH_SIZE     = 8
LR             = 1e-5        # MUST be 1/10 of Stage 1 LR
USE_OLORA      = True        # O-LoRA cross-adapter penalty
DEVICE         = "cuda"
REPO_PATH      = "/kaggle/input/calmsep-code"
NOISE_DIR      = "/kaggle/input/calmsep-8k/noise"
RIR_BANK       = "/kaggle/input/calmsep-8k/rirs/bank.json"
BUT_DIR        = "/kaggle/input/calmsep-8k/but-reverbdb"


In [ ]:
import sys, subprocess
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

cmd = [
    sys.executable, "-m", "train.stage4_joint",
    "--data-root", DATA_ROOT,
    "--checkpoint-dir", CHECKPOINT_DIR,
    "--stage1-dir", STAGE1_DIR,
    "--stage3-dir", STAGE3_DIR,
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--lr", str(LR),
    "--device", DEVICE,
    "--noise-dir", NOISE_DIR,
    "--rir-bank",  RIR_BANK,
    "--bf16",
]
if USE_OLORA:
    cmd.append("--use-olora")

print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=REPO_PATH)

In [ ]:
import os
expected = ["joint_reverb_adapter.pt", "joint_noise_adapter.pt",
            "joint_codec_adapter.pt", "joint_gate_net.pt",
            "joint_level2_analyzer.pt"]
for f in expected:
    path = os.path.join(CHECKPOINT_DIR, f)
    print(f"{f}: {'OK' if os.path.exists(path) else 'MISSING'}")

## After Stage 4

1. Run `eval_matrix.ipynb` to evaluate on the 8×4 fixed eval set.
2. Run `calibration_fit.ipynb` to fit temperature / confidence calibration.
3. Run `demo/app.py --calmsep` to test the full pipeline interactively.